In [6]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Load Dataset
df = pd.read_csv("garments_worker_productivity.csv")

df['department'] = df['department'].str.strip()
df['wip'] = df['wip'].fillna(0)

y_reg = df['actual_productivity']

df['Meets_Target'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)
y_clf = df['Meets_Target']


X = df.drop(columns=['date', 'actual_productivity', 'Meets_Target'])

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)


X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# ==========================================
# 5. REGRESSION: LinearRegression
# ==========================================
reg_model = LinearRegression()


t_start = time.perf_counter()
reg_model.fit(X_train_proc, y_reg_train)
reg_train_time = time.perf_counter() - t_start


t_start = time.perf_counter()
y_reg_pred = reg_model.predict(X_test_proc)
reg_pred_time = time.perf_counter() - t_start


mae = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
r2 = r2_score(y_reg_test, y_reg_pred)

# ==========================================
# 6. CLASSIFICATION: LogisticRegression
# ==========================================
clf_model = LogisticRegression(max_iter=1000, random_state=42)


t_start = time.perf_counter()
clf_model.fit(X_train_proc, y_clf_train)
clf_train_time = time.perf_counter() - t_start


t_start = time.perf_counter()
y_clf_pred = clf_model.predict(X_test_proc)
clf_pred_time = time.perf_counter() - t_start


accuracy = accuracy_score(y_clf_test, y_clf_pred)
precision = precision_score(y_clf_test, y_clf_pred, zero_division=0)
recall = recall_score(y_clf_test, y_clf_pred, zero_division=0)
f1 = f1_score(y_clf_test, y_clf_pred, zero_division=0)

# ==========================================
# 7. PRINT ALL RESULTS
# ==========================================


print("\n--- 1. Linear Regression (actual_productivity) ---")
print(f"MAE                   : {mae:.5f}")
print(f"RMSE                  : {rmse:.5f}")
print(f"R² Score              : {r2:.5f}")
print(f"Training Time         : {reg_train_time * 1000:.4f} ms")
print(f"Prediction Time       : {reg_pred_time * 1000:.4f} ms")

print("\n--- 2. Logistic Regression (Meets Target >= Targeted) ---")
print(f"Accuracy              : {accuracy:.5f}")
print(f"Precision             : {precision:.5f}")
print(f"Recall                : {recall:.5f}")
print(f"F1-Score              : {f1:.5f}")
print(f"Training Time         : {clf_train_time * 1000:.4f} ms")
print(f"Prediction Time       : {clf_pred_time * 1000:.4f} ms")
print("=" * 55)

# Optional: Store in DataFrame for quick copy-pasting to GitHub README tables
results_reg_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2", "Train Time (s)", "Predict Time (s)"],
    "Scikit-Learn Value": [f"{mae:.4f}", f"{rmse:.4f}", f"{r2:.4f}", f"{reg_train_time:.6f}", f"{reg_pred_time:.6f}"]
})

results_clf_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "Train Time (s)", "Predict Time (s)"],
    "Scikit-Learn Value": [f"{accuracy:.4f}", f"{precision:.4f}", f"{recall:.4f}", f"{f1:.4f}", f"{clf_train_time:.6f}", f"{clf_pred_time:.6f}"]
})

print("\nMarkdown Table for Regression:")
print(results_reg_df.to_markdown(index=False))

print("\nMarkdown Table for Classification:")
print(results_clf_df.to_markdown(index=False))


--- 1. Linear Regression (actual_productivity) ---
MAE                   : 0.10877
RMSE                  : 0.14813
R² Score              : 0.17358
Training Time         : 2.1369 ms
Prediction Time       : 0.1652 ms

--- 2. Logistic Regression (Meets Target >= Targeted) ---
Accuracy              : 0.74583
Precision             : 0.76606
Recall                : 0.94350
F1-Score              : 0.84557
Training Time         : 10.5150 ms
Prediction Time       : 0.1993 ms

Markdown Table for Regression:
| Metric           |   Scikit-Learn Value |
|:-----------------|---------------------:|
| MAE              |             0.1088   |
| RMSE             |             0.1481   |
| R2               |             0.1736   |
| Train Time (s)   |             0.002137 |
| Predict Time (s) |             0.000165 |

Markdown Table for Classification:
| Metric           |   Scikit-Learn Value |
|:-----------------|---------------------:|
| Accuracy         |             0.7458   |
| Precision        |

In [5]:
print("\nMarkdown Table for Classification:")
print(results_clf_df.to_markdown(index=False))


Markdown Table for Classification:
| Metric           |   Scikit-Learn Value |
|:-----------------|---------------------:|
| Accuracy         |             0.7458   |
| Precision        |             0.7661   |
| Recall           |             0.9435   |
| F1-Score         |             0.8456   |
| Train Time (s)   |             0.013749 |
| Predict Time (s) |             0.000368 |
